# From Zero to Claude: A Contract-Analysis Story

Learn how to analyze real legal contracts with Claude — from how LLMs work, to the two ways of calling Claude in production: the **Messages API** and the **Claude Agent SDK**.

This lab pairs with the guide **CCA Developer · M1 — MSO Foundations** (buildingai.in). Each chapter explains a concept, then a code cell lets you see it happen on a real contract.

| # | Chapter | You will learn |
|---|---|---|
| 1 | **Setup** | Install tools, load two contracts |
| 2 | **How LLMs Behave** | Tokens, context window, sampling, non-determinism, evaluation |
| 3 | **Model & Reasoning** | Pick a model, control thinking |
| 4 | **Prompting** | Zero-, one- and few-shot prompts, multi-turn |
| 5 | **Messages API** | SDK vs. REST, sync, streaming, async, batch |
| 6 | **Agent SDK vs. Messages API** | Same task, two interfaces |

### Prerequisites
Download both contracts (you'll upload them in Chapter 1):
- 📄 [AWS1.pdf](https://drive.google.com/file/d/1XSe2pSsGN1ssAbif92rvb80AnHb_Ni0F/view?usp=sharing) — short, ~12 pages
- 📚 [PROFRAC HOLDINGS, LLC credit agreement.pdf](https://drive.google.com/file/d/1UyOxeaEQsK5TFxXHI63PshoKmj0yjTmW/view?usp=sharing) — long, ~216 pages

## Chapter 1 — Setup

Install the libraries we need:

| Package | Used for |
|---|---|
| `anthropic` | Calling Claude (Messages API) |
| `PyMuPDF` | Extracting text from PDFs |
| `tiktoken` | Visualizing tokenization |
| `pyboxen` | Printing responses in readable boxes |

The Claude Agent SDK is installed later, in Chapter 6, when we need it.

In [ ]:
!pip install -q -U anthropic PyMuPDF tiktoken pyboxen

### 🔑 API key
Paste your **Anthropic** API key when prompted (it is not stored in the notebook). We also save it as `ANTHROPIC_API_KEY`, which the SDKs read automatically — including the Agent SDK in Chapter 6.

In [ ]:
import os, sys, base64, asyncio
from getpass import getpass

import anthropic
import fitz  # PyMuPDF
import tiktoken
from pyboxen import boxen

os.environ["ANTHROPIC_API_KEY"] = getpass("Anthropic API key: ")
client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY

# Model IDs used throughout the lab. Change them here and every chapter follows.
HAIKU = "claude-haiku-4-5"    # fast + cheap: extraction, classification
SONNET = "claude-sonnet-4-6"  # our default: general analysis


def show(title, text, color="cyan"):
    """Show text inside a titled, colored box."""
    sys.stdout.write(boxen(str(text), title=title, color=color, padding=1) + "\n")

### 📤 Upload the contracts
Run the next cell and select **both** PDFs: `AWS1.pdf` and `PROFRAC HOLDINGS, LLC credit agreement.pdf`.

In [ ]:
from google.colab import files
uploaded = files.upload()

Now extract the text, page by page. `contract_text` (the short AWS contract) is what we'll use for most of the notebook.

In [ ]:
SHORT_PDF = "/content/AWS1.pdf"
LONG_PDF = "/content/PROFRAC HOLDINGS, LLC credit agreement.pdf"


def extract_text(pdf_path):
    """Return the full text of a PDF, with one string per page."""
    with fitz.open(pdf_path) as doc:
        return [page.get_text() for page in doc]


short_pages = extract_text(SHORT_PDF)
long_pages = extract_text(LONG_PDF)

contract_text = "\n".join(short_pages)  # short contract as one string
long_text = "\n".join(long_pages)       # long contract as one string

show("Documents loaded", f"Short contract: {len(short_pages)} pages\nLong contract: {len(long_pages)} pages", "cyan")

## Chapter 2 — How LLMs Behave

Imagine an assistant where a user uploads a contract and asks:
> *"What is the termination notice period?"*

Before adding RAG or agents, we need to understand what happens when that request reaches an LLM. Five ideas:

1. **Tokens** — how text is processed
2. **Context window** — how much fits in one request
3. **Sampling** — how the next word is chosen
4. **Non-determinism** — why answers vary
5. **Evaluation** — how to test such a system

### 1. 🧩 Tokens
LLMs don't read characters or words — they read **tokens**, small chunks of text (a word, part of a word, or punctuation) mapped to integer IDs. Tokens are the foundation for reasoning about **cost, speed and context limits**.

Claude uses a **BPE (Byte Pair Encoding)** tokeniser. Common English words are usually one token; uncommon words, code symbols and non-Latin scripts are split into several.

| Content | Approximate tokens |
|---|---|
| 1 average English word | ~1.3 |
| 1 line of Python | ~5–12 |
| 1 character of Chinese / Japanese / Korean | ~2–3 |
| 1 JSON object with 5 fields | ~20–40 |
| 1 image (any size) | 1,000–3,000 (detail-dependent) |

> **Note:** structured output costs more than prose. A JSON wrapper spends tokens on `{`, `"`, `:` and `}` at every level of nesting — it adds up at scale.

Below we tokenize the first page of the contract. We use OpenAI's `cl100k_base` tokenizer purely to *illustrate* the idea; Claude has its own tokenizer, but the concept is identical.

In [ ]:
enc = tiktoken.get_encoding("cl100k_base")  # a public tokenizer, used only to visualise the idea
page = short_pages[0]
ids = enc.encode(page)                      # text -> list of integer token IDs

show("Original text (first 200 chars)", page[:200].replace("\n", " ") + "...", "yellow")
# Decode each ID back to text and join with "|" so you can see where the token boundaries fall
show("First 20 tokens, split with |", "|".join(enc.decode([t]) for t in ids[:20]), "magenta")
show("Token IDs (first 20)", ids[:20], "blue")
show("Size", f"Characters: {len(page):,} | Tokens: {len(ids):,} | ≈ {len(page) / len(ids):.1f} characters per token", "cyan")

**Takeaway:** ~4 characters ≈ 1 token in English (about 1.3 tokens per word). Tokens — not characters or pages — are what you pay for and what limits a request.

**Counting tokens with Claude.** Use the SDK's token counter *before* sending an expensive request — it is free to call. Let's compare the same idea written as prose, JSON, code and Chinese:

In [ ]:
samples = {
    "Prose":   "Either party may terminate this agreement with thirty days written notice.",
    "JSON":    '{"termination": {"by": ["customer", "provider"], "notice_days": 30, "form": "written"}}',
    "Python":  "if notice_days >= 30:\n    terminate(contract)",
    "Chinese": "任何一方均可提前三十天书面通知终止本协议。",
}

rows = []
for label, text in samples.items():
    n = client.messages.count_tokens(
        model=HAIKU, messages=[{"role": "user", "content": text}]
    ).input_tokens  # includes a few tokens of message overhead, so compare the numbers relatively
    rows.append(f"{label:<8} {len(text):>3} characters -> {n:>3} tokens")

show("Characters vs tokens", "\n".join(rows), "cyan")

Notice that JSON and Chinese need far more tokens per character than plain English prose. Same information, different bill.

### 2. 🪟 Context Window
The **context window** is the total number of tokens Claude can hold in one conversation — its *working memory*. It is a **shared resource**: the contract, your instructions **and** the model's answer all have to fit.

When you call the API you set `max_tokens`, the most output the model may generate. Input and output together must fit:

```text
input_tokens + max_tokens  ≤  context_window_size
```

For example, 180K tokens of input plus `max_tokens=32768` needs a window of at least ~213K tokens.

**KV cache.** The model keeps a Key-Value cache of the token representations it has already computed, so earlier tokens in a multi-turn conversation don't need to be recomputed. The cache is bounded by the context window — once a conversation outgrows it, *you* must decide what to drop.

Let's measure both contracts with Claude's token counter (free to call):

In [ ]:
token_counts = {}  # saved so the next cell can reuse the numbers
rows = []

for name, text in [("Short contract", contract_text), ("Long contract", long_text)]:
    n = client.messages.count_tokens(
        model=HAIKU, messages=[{"role": "user", "content": text}]
    ).input_tokens
    token_counts[name] = n
    rows.append(f"{name}: {n:,} tokens")

show("Token counts", "\n".join(rows), "cyan")

Now apply the rule: does `input_tokens + max_tokens` fit in the window? (Claude Haiku 4.5 has a 200K-token window; check the docs for the window of the model you use.)

In [ ]:
CONTEXT_WINDOW = 200_000  # tokens (Haiku 4.5). Larger windows are available on some models.
MAX_TOKENS = 1000         # the output budget we plan to request

rows = []
for name, n_input in token_counts.items():
    needed = n_input + MAX_TOKENS
    verdict = "✅ fits" if needed <= CONTEXT_WINDOW else "❌ too big"
    rows.append(f"{name}: {n_input:,} input + {MAX_TOKENS:,} output = {needed:,} of {CONTEXT_WINDOW:,} -> {verdict}")

show("Context window check", "\n".join(rows), "cyan")

#### 🗂️ Fill order and prompt caching
Because of the KV cache, **the order of your prompt matters**. Put stable content first and dynamic content last:

1. **System prompt** — stable, cached first
2. **Static documents / reference material** — e.g. the contract
3. **Conversation history** — older turns first
4. **Current user message** — dynamic, at the very end

Stable content at the front gets cached and reused; dynamic content at the end doesn't invalidate it.

> ⚠️ Don't prepend dynamic content (timestamps, user IDs) to the system prompt. It changes the start of the prompt on every request, invalidates the cache, and throws away the cost savings of prompt caching.

Below, the instructions and the contract are the stable prefix (marked with `cache_control`), and only the question changes. The usage numbers show the first call *writing* the cache and the second call *reading* it:

In [ ]:
def ask_cached(question):
    r = client.messages.create(
        model=SONNET,
        max_tokens=300,
        system=[
            # 1) stable instructions
            {"type": "text", "text": "You are a contract analyst. Answer only from the contract."},
            # 2) stable reference document - cache_control caches everything up to and including this block
            {"type": "text", "text": f"Contract:\n{contract_text}", "cache_control": {"type": "ephemeral"}},
        ],
        # 3) dynamic content goes LAST so it never invalidates the cached prefix
        messages=[{"role": "user", "content": question}],
    )
    u = r.usage
    show("Cache usage", f"cache written: {u.cache_creation_input_tokens or 0:>6,} | cache read: {u.cache_read_input_tokens or 0:>6,} | uncached input: {u.input_tokens:>6,}", "blue")
    return r.content[0].text


show("Call 1 · writes the cache", ask_cached("Who are the parties?"), "yellow")
show("Call 2 · reads the cache", ask_cached("What is the governing law?"), "green")
# If both 'cache written' and 'cache read' are 0, the prefix is shorter than the model's minimum cacheable size.

Separately from tokens, the API also limits **PDF uploads** (max 100 pages per request). Let's send each PDF directly to Claude as a document and see what happens:

In [ ]:
def pdf_block(path):
    """Wrap a PDF file as a base64 'document' block that Claude can read."""
    with open(path, "rb") as f:
        data = base64.standard_b64encode(f.read()).decode()
    return {"type": "document", "source": {"type": "base64", "media_type": "application/pdf", "data": data}}


for name, path in [("Short (≈12 pages)", SHORT_PDF), ("Long (≈216 pages)", LONG_PDF)]:
    try:
        r = client.messages.create(
            model=HAIKU,
            max_tokens=1000,
            messages=[{"role": "user", "content": [
                pdf_block(path),
                {"type": "text", "text": "Summarize the key terms of this contract."},
            ]}],
        )
        show(f"✅ {name}", r.content[0].text, "green")
    except anthropic.APIError as e:
        show(f"❌ {name} — rejected", e.message, "red")

**Takeaway:** Every request has limits (tokens *and* PDF pages). Long documents don't always fit, which is why techniques like **chunking and RAG** exist. For the rest of this notebook we send the contract as plain text.

### 3. 🎲 Sampling & Temperature
An LLM generates text **one token at a time**. At each step it assigns a probability to every possible next token, then **samples** one:

```text
Next token after "Notice period is ..."
"30"     60%     "thirty" 20%     "90" 10%     other 10%
```

**`temperature`** (0–1) controls that sampling:
- **Low (0)** → almost always picks the most likely token → consistent
- **High (1)** → more willing to pick less likely tokens → varied

The sampling parameters at a glance:

| Parameter | What it controls | Practical use |
|---|---|---|
| `temperature` | Width of the probability distribution (0–1) | Low (0–0.3) for factual / extraction tasks; high (0.7–1) for creative tasks |
| `top_p` | Nucleus sampling — cumulative probability cutoff | Tune **either** temperature **or** `top_p`, not both |
| `top_k` | Hard cap on how many candidate tokens are considered | Rarely tuned; stick with temperature |
| `max_tokens` | Maximum output length | Always set it explicitly — don't rely on a default |

We'll ask the same question 3 times at each temperature and compare:

In [ ]:
question = (
    "In one sentence, describe how either party can terminate this contract.\n\n" + contract_text
)

for temp in (0, 1):          # compare the most consistent setting with the most varied one
    for run in (1, 2, 3):    # same prompt, three runs: do the answers change?
        r = client.messages.create(
            model=HAIKU,
            max_tokens=150,                      # set the output budget explicitly
            extra_body={"temperature": temp},    # sampling control under test
            messages=[{"role": "user", "content": question}],
        )
        show(f"temperature={temp} · run {run}", r.content[0].text, "blue" if temp == 0 else "red")

### 4. 🔀 Non-determinism
At temperature 1 the same input produces different wording. Even at temperature 0, outputs are **near**-deterministic, not perfectly deterministic:

```text
Run 1 → "The notice period is 30 days."
Run 2 → "Either party must give 30 days' notice."
```
Different words, **same meaning**. Temperature is not the only source of variation. Even at `temperature=0`, outputs can still differ because of:
- **Hardware** — floating-point differences across server instances
- **Model updates** — a new model version changes behaviour
- **Token-budget pressure** — a model close to its `max_tokens` limit may abbreviate
- **Prompt caching** — subtle timing differences

So we can't test LLM apps by checking for an exact string. **Don't fight non-determinism — design around it:**
- Define expected **properties** of the output in your eval, not exact strings
- Use **structured output** (JSON schema or tool use) to constrain the format
- Test across **multiple samples** to catch edge-case behaviour
- **Pin the model ID** when you need stable behaviour (see Chapter 3)

### 5. ✅ Evaluation
Traditional software tests use **exact match**: `expected == actual`. For LLMs, many different answers can be correct, so we judge the answer on:

| Criterion | Question |
|---|---|
| **Correctness** | Is the extracted term right? |
| **Relevance** | Does it answer the question asked? |
| **Completeness** | Is anything important missing? |
| **Grounding** | Is it supported by the contract text? |

> Don't ask *"Did it produce exactly this sentence?"* — ask *"Is the answer correct?"*

## Chapter 3 — Model & Reasoning

Not every task needs the same intelligence:
- Extract a date → **simple**
- Summarize a clause → **moderate**
- Judge whether a termination clause is risky → **complex**

### The Claude model family
There's no "best" model — only trade-offs between **capability, latency and cost**. The family has four tiers:

| Model | Use case | Strengths | When *not* to use |
|---|---|---|---|
| **Claude Haiku 4.5** | High-volume, latency-sensitive tasks | Fastest and cheapest; good at extraction and classification | Complex reasoning, long-horizon tasks |
| **Claude Sonnet 5** | General purpose (the default) | Best balance of capability, speed and cost | After you've confirmed Sonnet fails your eval |
| **Claude Opus 5** | Complex research, long-horizon reasoning | Highest capability, extended thinking | High-volume use (cost-prohibitive) |
| **Claude Fable 5** | Narrative, roleplay, creative generation | Optimised for engaging long-form fiction | Factual Q&A, structured data extraction |

**Model selection rule**
1. **Start with Sonnet.**
2. Move **up** to Opus only if Sonnet fails your eval benchmarks on your specific task.
3. Move **down** to Haiku only if Sonnet passes but your cost or latency budget isn't met.

> ⚠️ *"This task sounds complex"* is not a valid reason to pick Opus. Run the eval on your actual task first — Sonnet handles most production workloads.

In this lab the `HAIKU` and `SONNET` variables from Chapter 1 pick the model, so you can swap in a newer one in one place.

### Alias vs. pinned model IDs
A model is identified by an ID string, and there are two kinds:
- **Alias** (e.g. `claude-sonnet-4-6`) — always resolves to the latest stable release of that model. Updates automatically; good for **development**.
- **Pinned** (the same name plus a release date) — resolves to one specific release. No surprise changes; **required once you have a validated eval suite**.

Let's ask the API which model IDs exist:

In [ ]:
from itertools import islice

# Each model is listed with its ID. Dated IDs are pinned releases; names without a date are aliases.
rows = [f"{m.id:<42} {m.display_name}" for m in islice(client.models.list(limit=20), 15)]

show("Available models", "\n".join(rows), "cyan")
show("In this lab", f"{HAIKU} and {SONNET}", "green")

### Reasoning (thinking)
Claude has two reasoning modes:
- **Basic (default)** — a direct response; reasoning happens implicitly while generating. Fast, and enough for the vast majority of tasks.
- **Extended thinking** — Claude first writes a scratchpad of reasoning tokens (a visible **thinking block**), then the final answer. This noticeably improves accuracy on multi-step problems.

| Task type | Extended thinking needed? |
|---|---|
| Simple Q&A, summarisation, extraction | No |
| Code generation for standard patterns | No |
| Multi-step math or logic proofs | Yes |
| Complex architecture trade-off analysis | Usually yes |
| Hard competitive-programming problems | Yes |

> **Cost:** thinking tokens count as **output tokens**. The `budget_tokens` value is a *ceiling* — the model uses what it needs, up to that limit. Turn thinking on only after basic mode has failed your eval.

You control it with the `thinking` parameter:

| Mode | Code | Behavior |
|---|---|---|
| **Off** | `thinking={"type": "disabled"}` | Answers directly |
| **Fixed budget** | `thinking={"type": "enabled", "budget_tokens": 5000}` | Thinks up to a budget you set |
| **Adaptive** ✨ | `thinking={"type": "adaptive"}` | Claude decides *if and how much* to think |

`output_config={"effort": "high"}` sets how hard Claude tries. Responses contain separate **thinking** and **text** blocks, so we can see both:

> Defaults differ by model: Sonnet 4.6 has thinking off; Sonnet 5 and Opus 5 use adaptive thinking.

In [ ]:
response = client.messages.create(
    model=SONNET,
    max_tokens=4000,
    thinking={"type": "adaptive"},         # let Claude decide whether (and how much) to think
    output_config={"effort": "high"},      # how hard it should try
    messages=[{"role": "user", "content": f"""
You are a contract risk analyst. For the AWS Customer Agreement below:
1. What termination rights does each party have?
2. What notice periods apply?
3. What happens to the customer's access and content after termination?
4. Is there significant risk for the customer?
5. Give the reviewer a concise recommendation.
Cite section numbers.

Contract:
{contract_text}
"""}],
)

# The response is a list of typed blocks: "thinking" (the scratchpad) and "text" (the final answer)
for block in response.content:
    if block.type == "thinking":
        show("🧠 Thinking", block.thinking, "purple")
    elif block.type == "text":
        show("💬 Answer", block.text, "green")

**What does thinking cost?** Run the same question with thinking **off** and with a **fixed budget**, then compare the output tokens. The extra tokens are the thinking block.

In [ ]:
risk_question = f"Is the termination clause risky for the customer? Answer in 3 sentences.\n\nContract:\n{contract_text}"

modes = [
    ("Thinking off", {"type": "disabled"}),
    ("Extended thinking (budget 4000)", {"type": "enabled", "budget_tokens": 4000}),  # ceiling, not a target
]

for label, thinking in modes:
    r = client.messages.create(
        model=SONNET,
        max_tokens=6000,   # must be larger than budget_tokens: thinking + answer share this budget
        thinking=thinking,
        messages=[{"role": "user", "content": risk_question}],
    )
    kinds = [b.type for b in r.content]
    answer = next(b.text for b in r.content if b.type == "text")
    show(label, f"blocks = {kinds} | output tokens = {r.usage.output_tokens}\n\n{answer}", "green")

## Chapter 4 — Prompting: Zero-, One- & Few-Shot

**Prompting** = giving the model instructions, context and (optionally) examples. The number of examples is the difference:

| Style | Examples | Use when |
|---|:-:|---|
| **Zero-shot** | 0 | The task is simple and self-explanatory |
| **One-shot** | 1 | You want to show the exact output format |
| **Few-shot** | 2–4 | There are different cases or edge cases (e.g. missing values) |

We'll extract the same key terms three ways. First, a small helper to avoid repeating the API call:

In [ ]:
def ask(prompt, model=HAIKU, max_tokens=1000, **kwargs):
    """Send one user prompt and return Claude's text answer."""
    r = client.messages.create(
        model=model, max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}], **kwargs,
    )
    return r.content[0].text

### 🎯 Zero-shot — instructions only

In [ ]:
zero_shot = f"""
Extract these key terms from the contract: parties, effective date, expiration date,
contract value, payment terms, termination terms, governing law.

Contract:
{contract_text}
"""

show("🎯 Zero-shot", ask(zero_shot))

### 🥇 One-shot — one example fixes the format

In [ ]:
one_shot = f"""
Extract the key terms as JSON, following the example. Return only JSON.

Example
Contract: "ABC Corp and XYZ Ltd agree to a 12-month term starting January 1, 2025, for $50,000,
payable net 30. Either party may terminate with 30 days' notice. Governed by New York law."
Output:
{{"parties": ["ABC Corp", "XYZ Ltd"], "effective_date": "2025-01-01", "expiration_date": "2025-12-31",
  "contract_value": 50000, "payment_terms": "Net 30", "termination_terms": "30 days' notice by either party",
  "governing_law": "New York"}}

Contract:
{contract_text}
Output:
"""

show("🥇 One-shot", ask(one_shot))

### 🏅 Few-shot — several examples cover edge cases (here: missing values → `null`)

In [ ]:
few_shot = f"""
Extract the key terms as JSON. Use null when a term is not stated. Return only JSON.

Example 1
Contract: "ABC Corp and XYZ Ltd agree to a 12-month term from January 1, 2025 for $50,000, net 30."
Output: {{"parties": ["ABC Corp", "XYZ Ltd"], "effective_date": "2025-01-01", "expiration_date": "2025-12-31",
          "contract_value": 50000, "payment_terms": "Net 30", "termination_terms": null, "governing_law": null}}

Example 2
Contract: "This agreement is effective March 3, 2024 between Acme Inc and Beta LLC. Either party may
terminate on 60 days' notice. Delaware law governs."
Output: {{"parties": ["Acme Inc", "Beta LLC"], "effective_date": "2024-03-03", "expiration_date": null,
          "contract_value": null, "payment_terms": null, "termination_terms": "60 days' notice by either party",
          "governing_law": "Delaware"}}

Example 3
Contract: "Services are provided by Orion Corp to Nova Ltd. Fees are USD 8,000 per month."
Output: {{"parties": ["Orion Corp", "Nova Ltd"], "effective_date": null, "expiration_date": null,
          "contract_value": null, "payment_terms": "USD 8,000 per month", "termination_terms": null, "governing_law": null}}

Contract:
{contract_text}
Output:
"""

show("🏅 Few-shot", ask(few_shot))

### What did we observe?
- **Zero-shot** — correct content, but the format is up to the model.
- **One-shot** — the output now follows our JSON structure.
- **Few-shot** — the model also handles *missing* values consistently (`null`).

**Guidelines for good examples**
- **Quality over quantity** — three excellent, diverse examples beat ten mediocre ones.
- **Cover the hard cases** — easy examples add no information; edge cases teach the boundary.
- **Match the format exactly** — if production input has typos, put a typo in an example.
- **Keep examples adjacent** to the instruction they illustrate.

> More examples ≠ always better. Ask: **"What is the smallest amount of guidance that gives the quality we need?"** If prompts keep growing, also consider a stronger model, more reasoning effort, or tighter output constraints.

### 💬 Multi-turn — conversation history
The Messages API is **stateless**: Claude only knows what is in the `messages` array you send. For a multi-turn conversation you pass the whole prior exchange each time. Use this for interactive assistants, and be deliberate about how much history you keep (context management is covered in the next lab).

In [ ]:
# Turn 1: a question about the contract
history = [{"role": "user", "content": f"Contract:\n{contract_text}\n\nWho are the parties?"}]
first = client.messages.create(model=HAIKU, max_tokens=300, messages=history)
show("Turn 1", first.content[0].text, "yellow")

# Keep Claude's reply in the history, then add a follow-up that only makes sense with the earlier turns
history.append({"role": "assistant", "content": first.content[0].text})
history.append({"role": "user", "content": "And which law governs it?"})

second = client.messages.create(model=HAIKU, max_tokens=300, messages=history)
show("Turn 2 (follow-up)", second.content[0].text, "green")
show("History", f"Messages sent on the 2nd call: {len(history)}", "cyan")  # history grows every turn -> so does the token bill

## Chapter 5 — Talking to Claude: The Messages API

Every call so far used the **Messages API**: you send messages, you get a response. What changes is *how* you wait for it:

| Pattern | Idea | Use when | What the user experiences |
|---|---|---|---|
| **Synchronous** | Send → wait → full response | Internal pipelines, short outputs, programmatic processing | Full wait; not suitable for interactive UIs |
| **Streaming** | Receive the answer piece by piece | Any user-facing output | First token in roughly half a second, then text appears progressively |
| **Async** | Send many requests concurrently | You need several answers fast | Seconds, all together |
| **Batch** | Submit a background job, collect results later | Non-urgent bulk workloads: nightly reports, large document sets, evals | Within a 24-hour window, at **50% lower cost** |

```text
Sync:   A ──wait──▶ resp A ; B ──wait──▶ resp B        (one after another)
Async:  A ─┐
        B ─┼──▶ Claude ──▶ responses                   (all at once)
        C ─┘
```

### SDK vs. raw REST
Under the hood every call is an HTTPS request to `/v1/messages`. The official Python and TypeScript **SDK** wraps it and adds:
- **Automatic retries** with exponential backoff (rate-limit errors are handled for you)
- **Streaming parsing** event by event, with no manual SSE handling
- **Type-safe** request and response objects
- **Helpers** such as token counting

Use the SDK for all production integrations. Reach for raw REST only when the SDK isn't available (an unusual runtime, or an edge function with size limits). Here is the same request both ways:

In [ ]:
import requests

payload = {
    "model": HAIKU,
    "max_tokens": 100,
    "messages": [{"role": "user", "content": "Say hello in one short sentence."}],
}

# Raw REST: you build the headers, parse the JSON and handle errors / retries yourself
raw = requests.post(
    "https://api.anthropic.com/v1/messages",
    headers={
        "x-api-key": os.environ["ANTHROPIC_API_KEY"],
        "anthropic-version": "2023-06-01",
        "content-type": "application/json",
    },
    json=payload,
    timeout=60,
)
raw.raise_for_status()
show("Raw REST", raw.json()["content"][0]["text"], "yellow")

# SDK: same call, but typed objects, and max_retries gives backoff on rate-limit / server errors
sdk_client = anthropic.Anthropic(max_retries=5)
r = sdk_client.messages.create(**payload)
show("SDK", r.content[0].text, "green")

### 1️⃣ Synchronous — blocks until the full answer arrives
Simplest to write, and fine for internal pipelines and scripts. For a user-facing UI the wait for the full response feels slow, so prefer streaming there.

In [ ]:
prompt = f"""
Identify the parties, effective date, expiration date, contract value and termination terms.

Contract:
{contract_text}
"""

response = client.messages.create(    # blocks here until the whole answer is ready
    model=SONNET,
    max_tokens=1000,
    messages=[{"role": "user", "content": prompt}],
)
show("1️⃣ Synchronous", response.content[0].text)

### 2️⃣ Streaming — text arrives as it is generated
For any user-facing output, stream. The first token shows up quickly and the user reads while the rest is generated. `client.messages.stream(...)` yields chunks through `stream.text_stream`. Watch the text appear live:

In [ ]:
with client.messages.stream(       # the SDK parses the server-sent events for us
    model=SONNET,
    max_tokens=1000,
    messages=[{"role": "user", "content": "Summarize the termination rights and notice periods.\n\n" + contract_text}],
) as stream:
    chunks = []
    for chunk in stream.text_stream:          # each chunk is a few tokens of text
        chunks.append(chunk)                  # a box can't be drawn incrementally, so collect the chunks

show("Streamed response", "".join(chunks), "green")

### 3️⃣ Async — don't block while waiting
`AsyncAnthropic` lets Python do other work while a request is in flight. Define the function with `async def` and wait for the result with `await`. (Colab allows `await` directly in a cell.)

In [ ]:
async_client = anthropic.AsyncAnthropic()  # reads ANTHROPIC_API_KEY


async def ask_async(prompt, max_tokens=500):
    r = await async_client.messages.create(
        model=SONNET, max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}],
    )
    return r.content[0].text


show("3️⃣ Async", await ask_async("Summarize the termination rights and risks.\n\n" + contract_text))

### 4️⃣ Many async calls in parallel
`asyncio.gather` runs all five requests **at the same time**, so total time ≈ the slowest request, not the sum.

In [ ]:
topics = ["contract parties", "contract value", "effective date", "termination terms", "governing law"]

answers = await asyncio.gather(
    *(ask_async(f"Identify the {t}.\n\nContract:\n{contract_text}") for t in topics)
)

for topic, answer in zip(topics, answers):
    show(f"4️⃣ {topic}", answer)

### 5️⃣ Message Batches — background jobs for evals
Say we want the **same 3 questions** answered for **both contracts** → 2 × 3 = **6 independent requests**. Instead of sending them one by one, submit them as one **batch**, then check back for the results. Batches have a **24-hour processing window** and cost **50% less**, so they suit non-urgent bulk work such as nightly reports or large document sets.

```text
2 contracts × 3 questions ──▶ Batch ──▶ poll status ──▶ fetch results
```

> 💡 For high-volume overnight jobs the cheapest combination is **Haiku + the Batches API**: the cheapest model, the batch discount, and no latency requirement.

**Step 1 — build and submit the 6 requests**

In [ ]:
questions = [
    "What is the total contract value?",
    "What are the payment terms?",
    "What are the termination conditions?",
]

batch_requests = [
    {
        "custom_id": f"{name}_q{i}",  # lets us match each result to its request
        "params": {
            "model": SONNET,
            "max_tokens": 500,
            "messages": [{"role": "user", "content":
                f"Contract:\n{text}\n\nQuestion: {q}\nAnswer only using information from the contract."}],
        },
    }
    for name, text in [("12_page", contract_text), ("216_page", long_text)]
    for i, q in enumerate(questions, 1)
]

batch = client.messages.batches.create(requests=batch_requests)
show("Batch submitted", f"{len(batch_requests)} requests · batch id: {batch.id}", "yellow")

**Step 2 — wait for completion, then read the results** (batches run in the background and can take a few minutes)

In [ ]:
import time

while (status := client.messages.batches.retrieve(batch.id)).processing_status != "ended":
    show("Batch status", status.processing_status, "yellow")
    time.sleep(5)

for item in client.messages.batches.results(batch.id):
    if item.result.type == "succeeded":
        show(item.custom_id, item.result.message.content[0].text, "green")
    else:
        show(item.custom_id, f"Request {item.result.type}", "red")

### Sync vs. streaming vs. async vs. batch

| | Question you're asking | Result arrives |
|---|---|---|
| **Sync** | "I need this answer now, in a script." | Immediately, after the full wait |
| **Streaming** | "A user is watching." | Progressively, first token in ~0.5 s |
| **Async** | "I need several answers, fast." | Concurrently, in seconds |
| **Batch** | "I have many independent jobs and can wait." | Later, in the background (within 24 h, at 50% lower cost) |

## Chapter 6 — Claude Agent SDK vs. the Messages API

So far **we** wrote the prompt, called `client.messages.create(...)`, and got one response back. That's the **Messages API**: low-level, request in → response out.

The **Claude Agent SDK** works differently: you give Claude a *task*, and it can loop over multiple turns, use tools (files, bash, search, your own tools), and stream back a sequence of typed messages.

| | Messages API | Claude Agent SDK |
|---|---|---|
| **Call** | `client.messages.create()` | `query()` |
| **Shape** | 1 request → 1 response | 1 task → stream of messages, many turns |
| **Tools** | You run the tool loop yourself | Built-in loop and tools |
| **Best for** | Single-shot generation and extraction | Multi-step, tool-using workflows |

### Install
The SDK runs on the Claude Code CLI (needs Node.js, already in Colab). It uses the `ANTHROPIC_API_KEY` we set in Chapter 1.

In [ ]:
!npm install -g @anthropic-ai/claude-code
!pip install -q -U claude-agent-sdk

### Same task, two interfaces

**A) Messages API** — one call, one response object:

In [ ]:
task_prompt = f"""
Analyze this contract and identify: 1. parties, 2. effective date, 3. contract value, 4. termination terms.

Contract:
{contract_text}
"""

response = client.messages.create(
    model=SONNET, max_tokens=1000,
    messages=[{"role": "user", "content": task_prompt}],
)
show("📨 Messages API", response.content[0].text)  # the answer is right here

**B) Claude Agent SDK** — `query()` is async and yields a **stream** of messages. Claude's answer is inside `AssistantMessage` → `TextBlock`, so we have to pick it out of the stream. (The same loop would carry tool calls if Claude decided to use tools.)

In [ ]:
from claude_agent_sdk import query, ClaudeAgentOptions, AssistantMessage, TextBlock


async def ask_agent(prompt):
    options = ClaudeAgentOptions(model=SONNET, max_turns=1)  # single turn, for a fair comparison
    text = ""
    async for message in query(prompt=prompt, options=options):
        if isinstance(message, AssistantMessage):
            text += "".join(b.text for b in message.content if isinstance(b, TextBlock))
    return text


show("🤖 Claude Agent SDK", await ask_agent(task_prompt), "magenta")

### Which should you use?
Both answered the same question, but the Agent SDK needed more code because it is built for **multi-step, tool-using agents**.

- **Messages API** → *"I have one prompt and want one answer."* (Chapters 2–5)
- **Agent SDK** → *"Go through this folder of contracts, flag non-standard termination clauses, and write a summary file."* — Claude decides what to do next.

## 🏁 Cheat Sheet

| Concept | One-line takeaway |
|---|---|
| **Tokens** | The unit models read and you pay for: ~1.3 tokens per English word (~4 characters); code, JSON and non-English text cost more |
| **Context window** | `input_tokens + max_tokens ≤ window` — budget both; long docs need chunking/RAG |
| **Fill order / caching** | Stable content first, dynamic content last; never put timestamps or IDs at the front of the system prompt |
| **Temperature** | Low = consistent, high = varied; tune temperature **or** `top_p`, not both |
| **Non-determinism** | Even `temperature=0` isn't perfectly deterministic → test for *correctness*, not exact match |
| **Model choice** | Start with Sonnet; move to Opus only if it fails your eval, to Haiku only if cost/latency demands it |
| **Alias vs. pinned ID** | Alias for development; pin the ID once you have a validated eval suite |
| **Extended / adaptive thinking** | Thinking tokens are output tokens — enable it only when basic mode fails your eval |
| **Zero / one / few-shot** | Add only as many examples as the task needs; cover the hard cases |
| **Multi-turn** | The API is stateless — you resend the history, and the bill grows with it |
| **SDK vs. REST** | Use the SDK in production (retries, streaming, types); raw REST only when it's unavailable |
| **Sync · Stream · Async · Batch** | Scripts · user-facing · concurrent · bulk and non-urgent (24 h, 50% cheaper) |
| **Messages API vs. Agent SDK** | One answer vs. a multi-step, tool-using agent |